# Inspect demand curves

Load the cached demand curve tables (`artifacts/curves/{train,test}_demand.pkl`)
and sanity-check the (qty, price) step-function pairs.

Each row of the table is one auction `(Trading Date, Interval Number)` with
`curve_qty` and `curve_price` as numpy arrays of equal length — the list of
step-function pairs that defines the demand curve.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aemo_curves.data import load_curve_table

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1 — Load train + test demand tables

In [2]:
train = load_curve_table('train', 'demand')
test = load_curve_table('test', 'demand')

print(f'train: {len(train):,} auctions, '
      f'{train["Trading Date"].min()} -> {train["Trading Date"].max()}')
print(f'test:  {len(test):,} auctions, '
      f'{test["Trading Date"].min()} -> {test["Trading Date"].max()}')
print('columns:', list(train.columns))
train.head(3)

train: 58,368 auctions, 2021-01-01 -> 2024-04-30
test:  17,232 auctions, 2024-05-01 -> 2025-04-24
columns: ['Trading Date', 'Interval Number', 'curve_qty', 'curve_price', 'n_offers', 'total_quantity', 'mean_price']


,Trading Date,Interval Number,curve_qty,curve_price,n_offers,total_quantity,mean_price
0,2021-01-01,1,"[0.721, 10.721, 13.552, 16.552, 18.552, 24.552...","[49.83, 45.32, 27.99, 27.73, 27.61, 27.59, 27....",23,681.397,-267.910870
1,2021-01-01,2,"[0.721, 10.721, 21.500999999999998, 31.5009999...","[49.6, 45.32, 26.45, 0.0, -8.8, -43.7, -47.62,...",18,643.671,-350.033889
2,2021-01-01,3,"[0.721, 10.721, 22.631, 32.631, 32.885, 161.62...","[48.06, 45.32, 26.45, 0.0, -8.8, -43.7, -47.62...",19,615.878,-334.377368


## 2 — Per-row shapes and the (qty, price) pairs

Confirm `curve_qty` and `curve_price` have the same length on every row,
and peek at one auction's pairs.

In [ ]:
lens_qty = train['curve_qty'].map(len)
lens_price = train['curve_price'].map(len)
assert (lens_qty == lens_price).all(), 'qty/price length mismatch on some row'

print('curve length (#steps) — train:')
print(lens_qty.describe().round(1))

row = train.iloc[0]
pairs = list(zip(row['curve_qty'], row['curve_price']))
print(f"\nFirst auction: {row['Trading Date'].date()} interval {row['Interval Number']}")
print(f"n_steps={len(pairs)}, n_bids={row['n_offers']}, total_qty={row['total_quantity']:.1f}")
print('first 8 (qty, price) pairs:')
for q, p in pairs[:8]:
    print(f'  ({q:9.3f}, {p:9.2f})')

## 3 — Plot a few random demand curves

Demand curves are downward step-functions: highest-value bids accepted first.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(train), size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharey=False)
for ax, i in zip(axes.flat, idx):
    r = train.iloc[i]
    ax.step(r['curve_qty'], r['curve_price'], where='post')
    ax.set_title(f"{r['Trading Date'].date()}  int {r['Interval Number']}", fontsize=9)
    ax.set_xlabel('cumulative qty (MWh)')
    ax.set_ylabel('price ($/MWh)')
plt.tight_layout(); plt.show()

## 4 — Daily / interval coverage

There should be ~48 intervals per day. Spot any holes.

In [ ]:
per_day = test.groupby('Trading Date').size()
print('intervals per day on test:')
print(per_day.describe().round(1))
missing = per_day[per_day < 48]
print(f'\ndays with fewer than 48 intervals: {len(missing)}')
missing.head(10)

## 5 — Optional: export pairs to CSV (long format)

Uncomment to write one row per `(date, interval, qty, price)` step.

In [ ]:
# long = test[['Trading Date', 'Interval Number', 'curve_qty', 'curve_price']].explode(
#     ['curve_qty', 'curve_price']
# )
# long.to_csv('test_demand_pairs.csv', index=False)
# print('wrote', len(long), 'rows -> test_demand_pairs.csv')